# Use LLM to enrich parsed templates with corpus‑level and sample‑level semantic descriptions and reasoning traces

This idea is taken from the paper ["Log Anomaly Detection with Large Language Models via Knowledge-Enriched Fusion"](https://arxiv.org/html/2512.11997v1)


In [ ]:
import os, sys
import json
from dotenv import load_dotenv
load_dotenv()  # load environment variables from .env file

sys.path.insert(0, os.path.abspath(".."))  # project root

In [ ]:
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")

# Auto-pick the newest *_hdfs_templates.json produced by the Parser notebook.
# Override to pin a specific run:  templates_file = PROCESSED_DIR / "20260407_1423_hdfs_templates.json"
candidates = sorted(PROCESSED_DIR.glob("*_hdfs_templates.json"))
if not candidates:
    raise FileNotFoundError(f"No *_hdfs_templates.json found in {PROCESSED_DIR}")

templates_file = candidates[-1]          # lexicographic sort → newest YYYYMMDD_HHMM prefix last
RUN_TAG = templates_file.stem.replace("_hdfs_templates", "")

print(f"Using run  : {RUN_TAG}")
print(f"Input file : {templates_file}")

with open(templates_file, "r") as f:
    data = json.load(f)

templates = [entry["template"] for entry in data]

print(f"\nLoaded {len(templates)} templates")
print("First 3:")
for t in templates[:3]:
    print(" ", t)


In [ ]:
from src.enricher import Enricher

MISTRAL_LARGE_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT_MISTRAL_LARGE")
MISTRAL_SMALL_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT_MISTRAL_SMALL")

enricher_large = Enricher(MISTRAL_LARGE_DEPLOYMENT)
enricher_small = Enricher(MISTRAL_SMALL_DEPLOYMENT)

# result = enricher.enrich_corpus_hdfs(template)

# print(result)

In [ ]:
enriched_with_large = []

for template in templates:
	enriched = enricher_large.enrich_corpus_hdfs(template)
	enriched_with_large.append(enriched)

for index, enriched in enumerate(enriched_with_large):
	data[index]['enriched_large'] = enriched

In [ ]:
enriched_with_small = []

for template in templates:
	enriched = enricher_small.enrich_corpus_hdfs(template)
	enriched_with_small.append(enriched)

for index, enriched in enumerate(enriched_with_small):
	data[index]['enriched_small'] = enriched

In [ ]:
for i, entry in enumerate(data):
    if i < len(enriched_with_large):
        entry["enriched_large"] = enriched_with_large[i].model_dump_json()
    if i < len(enriched_with_small):
        entry["enriched_small"] = enriched_with_small[i].model_dump_json()

path_to_output = PROCESSED_DIR / f"{RUN_TAG}_hdfs_templates_enriched.json"
with open(path_to_output, "w") as f:
    json.dump(data, f, indent=2)

print(f"Saved {len(data)} records → {path_to_output}")
